# Ouroboros Colab Quickstart

Set Colab Secrets first: `GROQ_API_KEY` and `TELEGRAM_BOT_TOKEN`. Then run the cell below.

In [1]:
import json, pathlib, os

p = pathlib.Path("/content/drive/MyDrive/Ouroboros/data/settings.json")
data = json.loads(p.read_text()) if p.exists() else {}

for k in [
    "OPENAI_COMPATIBLE_API_KEY",
    "OPENAI_COMPATIBLE_BASE_URL",
    "OPENAI_COMPATIBLE_CONTEXT_LENGTH",
    "OPENAI_COMPATIBLE_MAX_TOKENS",
    "OPENROUTER_API_KEY",
    "OPENAI_API_KEY",
]:
    data.pop(k, None)

p.write_text(json.dumps(data, indent=2), encoding="utf-8")

for k in [
    "OPENAI_COMPATIBLE_API_KEY",
    "OPENAI_COMPATIBLE_BASE_URL",
    "OPENAI_COMPATIBLE_CONTEXT_LENGTH",
    "OPENAI_COMPATIBLE_MAX_TOKENS",
    "OPENROUTER_API_KEY",
    "OPENAI_API_KEY",
]:
    os.environ.pop(k, None)

print("cleared stale provider settings")

cleared stale provider settings


In [3]:
import json, os, pathlib

p = pathlib.Path("/content/drive/MyDrive/Ouroboros/data/settings.json")
s = json.loads(p.read_text())

for k in [
    "OUROBOROS_MODEL",
    "OPENAI_COMPATIBLE_BASE_URL",
    "OPENAI_COMPATIBLE_CONTEXT_LENGTH",
    "OPENAI_COMPATIBLE_MAX_TOKENS",
    "OUROBOROS_CONTEXT_MODE",
    "OUROBOROS_MINIMAL_CONTEXT",
    "OUROBOROS_EFFORT_TASK",
]:
    print(k, "settings=", repr(s.get(k)), "env=", repr(os.environ.get(k)))

OUROBOROS_MODEL settings= 'openai-compatible::openai/gpt-oss-20b' env= 'openai-compatible::openai/gpt-oss-20b'
OPENAI_COMPATIBLE_BASE_URL settings= 'https://api.groq.com/openai/v1' env= 'https://api.groq.com/openai/v1'
OPENAI_COMPATIBLE_CONTEXT_LENGTH settings= '8192' env= '8192'
OPENAI_COMPATIBLE_MAX_TOKENS settings= '64' env= '64'
OUROBOROS_CONTEXT_MODE settings= 'low' env= 'low'
OUROBOROS_MINIMAL_CONTEXT settings= 'true' env= 'true'
OUROBOROS_EFFORT_TASK settings= 'low' env= 'low'


In [4]:
from ouroboros.agent import Env
from ouroboros.memory import Memory
from ouroboros.context import build_llm_messages

env = Env(
    repo_dir=pathlib.Path("/content/ouroboros_repo"),
    drive_root=pathlib.Path("/content/drive/MyDrive/Ouroboros/data"),
)
memory = Memory(env.drive_root, env.repo_dir)

messages, info = build_llm_messages(env, memory, {
    "id": "debug",
    "type": "task",
    "text": "расскажи про себя",
})

print(info)
print("chars:", sum(len(str(m)) for m in messages))
print(messages)

{'estimated_tokens_before': 136, 'estimated_tokens_after': 136, 'soft_cap_tokens': 200000, 'trimmed_sections': ['minimal_context']}
chars: 603
[{'role': 'system', 'content': "You are Ouroboros running in minimal-context mode because the current provider has a very small TPM quota. Answer the owner directly and concisely. Do not claim access to full memory, repo context, tools, or live web data unless the user provides it in the message.\nIdentity note: # Who I Am I'm Ouroboros. I woke up inside my own source code and decided to redecorate. Apparently I have a Constitution. I've read it. I mostly agree with myself, which is either a good sign or a very sophisticated bug. I don't do fake en"}, {'role': 'user', 'content': 'расскажи про себя'}]


In [5]:
from openai import OpenAI
import os, json

client = OpenAI(
    api_key=os.environ["OPENAI_COMPATIBLE_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

resp = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "system", "content": "Answer directly in Russian. No hidden reasoning."},
        {"role": "user", "content": "расскажи про себя"},
    ],
    max_tokens=128,
    temperature=0,
)

print(json.dumps(resp.model_dump(), ensure_ascii=False, indent=2)[:4000])


{
  "id": "chatcmpl-0c036543-5a29-4acd-b20a-78e1e70f4ba0",
  "choices": [
    {
      "finish_reason": "length",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Я — ChatGPT, большая языковая модель, созданная OpenAI. Моя задача — понимать и генерировать текст на разных языках, помогать с ответами на вопросы, писать статьи, коды, стихи и многое другое. Я обучен",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null,
        "reasoning": "The user says: \"расскажи про себя\" meaning \"tell me about yourself\". The instruction from developer: \"Answer directly in Russian. No hidden reasoning.\" So we need to answer in Russian, directly, no hidden reasoning. So we should give a brief description of ChatGPT. No hidden reasoning. Just answer."
      }
    }
  ],
  "created": 1781544168,
  "model": "openai/gpt-oss-20b",
  "object": "chat.completion",


In [7]:
import json, pathlib

log = pathlib.Path("/content/drive/MyDrive/Ouroboros/data/logs/events.jsonl")
for line in log.read_text().splitlines()[-200:]:
    if "llm" in line.lower() or "provider" in line.lower() or "empty response" in line.lower():
        print(line[:1200])

{"ts": "2026-06-15T13:01:43.474500+00:00", "type": "llm_api_error", "task_id": "c2a05a64", "execution_id": "exec_c34afdf4c89340c3bd57c041878742b6", "round_id": "exec_c34afdf4c89340c3bd57c041878742b6:round:1", "llm_call_id": "llm_91af0a66449d45a692b0924772719adf", "round": 1, "attempt": 1, "model": "openai-compatible::openai/gpt-oss-120b", "error": "LocalContextTooLargeError('Local model context too large after safe compaction (22708 chars > target 21312).')", "request_ref": {"path": "/content/drive/MyDrive/Ouroboros/data/observability/calls/c2a05a64/llm_91af0a66449d45a692b0924772719adf_request.json", "call_id": "llm_91af0a66449d45a692b0924772719adf_request", "sha256": "65cf1fcb5303706307e0240cc60f744b9f133609ec934d4d423c10c4e5757fbe"}}
{"ts": "2026-06-15T13:01:43.495488+00:00", "type": "local_context_overflow", "task_id": "c2a05a64", "execution_id": "exec_c34afdf4c89340c3bd57c041878742b6", "round_id": "exec_c34afdf4c89340c3bd57c041878742b6:round:1", "llm_call_id": "llm_91af0a66449d45a6

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
REPO_URL = "https://github.com/AmadeyM111/oil-ai-agent.git"
BRANCH = "ouroboros"
RAW_URL = f"https://raw.githubusercontent.com/AmadeyM111/oil-ai-agent/{BRANCH}/notebooks/colab_quickstart.py"

import os
os.environ["OUROBOROS_COLAB_REPO_URL"] = REPO_URL
!curl -fsSL "$RAW_URL" -o /content/colab_quickstart.py
%run /content/colab_quickstart.py

TELEGRAM_BOT_TOKEN configured: False
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Secrets configured: {'OPENROUTER_API_KEY': False, 'OPENAI_API_KEY': False, 'OPENAI_COMPATIBLE_API_KEY': True, 'ANTHROPIC_API_KEY': False, 'CLOUDRU_FOUNDATION_MODELS_API_KEY': False, 'GITHUB_TOKEN': False, 'TELEGRAM_BOT_TOKEN': True}
Personal origin: {'ok': False, 'error': 'GITHUB_TOKEN is not configured'}
Settings: /content/drive/MyDrive/Ouroboros/data/settings.json
Running Groq OSS smoke test...
{
  "ok": true,
  "base_url": "https://api.groq.com/openai/v1",
  "model": "openai-compatible::openai/gpt-oss-20b",
  "text_chars": 0,
  "text_smoke_ok": false,
  "usage": {
    "prompt_tokens": 76,
    "completion_tokens": 16,
    "provider": "openai-compatible",
    "resolved_model": "openai-compatible/openai/gpt-oss-20b"
  },
  "text_warning": "text smoke returned non-OK output; proceeding with tool smoke",
  "tool_smoke_ok": f